# SINCON 2026 핫식스 — 전체 실행 노트북

이 노트북을 위에서부터 끝까지 실행하면 제공 데이터에서 최종 예측 `submission_final_main.csv` (Public 0.143043)까지 차례로 다시 만들고, 제출 파일과 비교한다.

| 순서 | 폴더 (`code/`) | 하는 일 | 출력 |
|---|---|---|---|
| 1 | `1_base_models` | 기본 모델들의 예측을 결합하고 하위 보정을 적용 | `submission_clean_refined.csv` |
| 2 | `2_corrections` | 과거 검증을 통과한 보정 6개 | `submission_night_v4_main.csv` |
| 2b, 2c | `2b_monday`, `2c_cup_walk` | 3단계가 읽는 두 구성 입력을 다시 만들어 보관본과 대조 | `submission_monday_main.csv`, `submission_night_v4_main_cup_walk.csv` |
| 3 | `3_stack` | 보정 7개 단계 | `submission_xgb7_w10_on_curvature.csv` |
| 4 | `4_rail` | 다른 노선 자료를 쓰는 세 층 | `submission_rail_factors_k30_hour_on_xgb7.csv` = **`submission_final_main.csv`** |

- 각 단계의 계산 코드는 해당 폴더의 노트북과 `.py` 모듈에 있다. 이 노트북은 단계마다 그 폴더의 노트북 코드 셀을 **별도 Python 프로세스에서 순서대로 실행**하고, 출력을 아래에 그대로 보여 준다. 단계끼리 모듈 이름이 겹치기 때문에 프로세스를 나눴다.
- 기본 모드(`RUN_MODE = "assemble"`)는 보관한 모델별 예측에서 조립한다. 난수를 쓰지 않고 약 1분 걸린다. 모델을 처음부터 다시 학습하는 방법은 `docs/RETRAINING.md`.
- 필요한 것: Python 3.12 (또는 3.11), `code/requirements.txt` 의 패키지, `ipykernel`. GPU는 필요 없다.
- 방법 설명은 `docs/METHOD.md`, 데이터 출처와 산출물은 `docs/DATA_AND_ARTIFACTS.md`, 평가 기간 자료의 사용에 관한 사실 기재는 `docs/DISCLOSURE.md`.

In [1]:
# ===== 설정 =====
PROJECT = ""           # 제출 폴더(이 노트북과 code/, predictions/ 가 있는 곳). 비우면 현재 폴더에서 찾는다.
OUT = ""               # 결과를 쓸 폴더. 비우면 <PROJECT>/run_outputs
RUN_MODE = "assemble"  # "assemble": 보관한 모델 예측에서 조립.  "full": 1단계 모델을 train.csv 부터 다시 학습(수 시간, 아래 1단계 설명 참고)
DEVICE = "cpu"         # 1단계 재학습에 쓸 장치: "cpu" 또는 "auto"(CUDA가 있으면 사용)
PIP_NO_INDEX = False   # True: 1단계의 pip 설치 확인을 인터넷 없이 한다(패키지를 미리 설치해 둔 경우)

## 0. 준비: 경로, 환경, 실행 도구

In [2]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

def find_project():
    for base in [Path(PROJECT)] if PROJECT else [Path.cwd(), *Path.cwd().parents][:4]:
        if (base / "code/1_base_models/payload_manifest.json").is_file() and (base / "predictions/submission_final_main.csv").is_file():
            return base.resolve()
    raise FileNotFoundError("첫 셀의 PROJECT 에 제출 폴더 경로를 적는다.")

project = find_project(); CODE = project / "code"
out = Path(OUT).resolve() if OUT else project / "run_outputs"
assert CODE not in out.parents and out != CODE, "OUT 은 code/ 폴더 밖에 둔다."
(out / "logs").mkdir(parents=True, exist_ok=True)

# 단계 노트북의 코드 셀을 별도 프로세스에서 순서대로 실행하는 도구.
# 첫 코드 셀(설정 셀)을 실행한 직후에 overrides 의 값을 넣어 입력·출력 경로를 지정한다. 셀의 마지막 식은 화면에 출력한다.
RUNNER = r"""
import ast, json, os, sys
nb_path, overrides = os.path.abspath(sys.argv[1]), json.loads(sys.argv[2])
os.chdir(os.path.dirname(nb_path)); sys.path.insert(0, os.getcwd()); sys.dont_write_bytecode = True
def show(value):
    if value is None: return
    try:
        import pandas as pd
        if isinstance(value, (pd.DataFrame, pd.Series)): print(value.to_string()); return
    except Exception: pass
    print(repr(value))
ns = {"__name__": "__main__", "display": show}
cells = [c for c in json.load(open(nb_path, encoding="utf-8"))["cells"] if c["cell_type"] == "code"]
for i, cell in enumerate(cells, 1):
    print(f"\n--- 코드 셀 {i}/{len(cells)} ---", flush=True)
    name = f"{os.path.basename(nb_path)}:cell{i}"; tree = ast.parse("".join(cell["source"]), name)
    last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None
    exec(compile(tree, name, "exec"), ns)
    if last is not None: show(eval(compile(ast.Expression(last.value), name, "eval"), ns))
    if i == 1: ns.update(overrides)
    sys.stdout.flush()
"""
(out / "run_notebook_cells.py").write_text(RUNNER, encoding="utf-8")
S, T = {}, {}     # 단계별 출력 경로, 걸린 시간(초)

def run(name, cmd, cwd=None, env=None):
    """명령을 실행하고 출력을 그대로 보여 준다. 로그는 <out>/logs/<name>.log 에 남긴다."""
    started = time.time(); environment = {**os.environ, "PYTHONUTF8": "1", "PYTHONDONTWRITEBYTECODE": "1", **{k: str(v) for k, v in (env or {}).items()}}
    done = subprocess.run([str(c) for c in cmd], cwd=cwd, env=environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    (out / "logs" / f"{name}.log").write_text(done.stdout, encoding="utf-8"); print(done.stdout)
    assert done.returncode == 0, f"{name} 단계가 실패했다. 위 출력의 오류를 본다."
    T[name] = round(time.time() - started, 1); print(f"[{name}] {T[name]}초"); return started

def run_notebook(name, notebook, overrides=None, env=None):
    return run(name, [sys.executable, "-B", out / "run_notebook_cells.py", notebook, json.dumps({k: str(v) for k, v in (overrides or {}).items()})], env=env)

def newest(patterns, since):
    hits = [p for pattern in patterns for p in glob.glob(str(pattern)) if os.path.getmtime(p) >= since - 1]
    assert hits, f"출력을 찾지 못했다: {patterns}"; return Path(max(hits, key=os.path.getmtime))

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""): h.update(block)
    return h.hexdigest()

def compare(a, b):
    """두 제출 CSV의 ID·순서가 같은지 확인하고 log1p 예측의 최대 차이를 돌려준다."""
    x, y = pd.read_csv(a), pd.read_csv(b)
    assert list(x.columns) == list(y.columns) and len(x) == len(y) and (x.iloc[:, 0] == y.iloc[:, 0]).all(), f"ID가 다르다: {a} / {b}"
    return float(np.abs(np.log1p(x.iloc[:, -1].to_numpy(float)) - np.log1p(y.iloc[:, -1].to_numpy(float))).max())

import numpy as np, pandas as pd
print("python", platform.python_version(), "|", platform.platform(), "| numpy", np.__version__, "| pandas", pd.__version__)
print("제출 폴더:", project); print("결과 폴더:", out); print("모드:", RUN_MODE, "| 장치:", DEVICE)

python 3.12.15 | macOS-26.6-arm64-arm-64bit | numpy 2.5.4 | pandas 3.1.1
제출 폴더: <실행 폴더>
결과 폴더: <실행 폴더>/run_outputs
모드: assemble | 장치: cpu


## 1. 1단계 — 기본 모델 결합과 하위 보정 (`code/1_base_models/SINCON_final.ipynb`)

- 모델: 팀 LightGBM(시드 3)·임베딩 MLP(시드 30), 생활이동·버스 LightGBM, 그래프·수요 혼합 신경망, 가법 시간대 모형, 경기 모형, 축제·주거 보정, 시간별 XGBoost, 벡터 XGBoost, TabICL(공개 사전학습 모델), 행사 응답. 시드와 설정은 `docs/RETRAINING.md` 1절.
- `assemble`: `payload/` 에 보관한 모델별 예측을 읽어 결합(스택)과 하위 보정, 행사 보정을 다시 계산한다.
- `full`: 위 모델을 `train.csv` 부터 다시 학습한다. 로컬 CPU에서 약 5.5시간 걸렸다. 신경망은 기기에 따라 값이 달라지므로 결과가 보관본과 같지 않고, 그 경우 2단계의 일치 검사(1e-9)에서 멈춘다.
- 첫 셀은 고정 버전의 패키지를 `pip` 로 확인·설치한다.

In [3]:
env = {"SINCON_LOCAL_ROOT": CODE / "1_base_models", "SINCON_RUN_MODE": RUN_MODE, "SINCON_DEVICE": DEVICE, "SINCON_THREADS": 4}
if PIP_NO_INDEX: env["SINCON_PIP_NO_INDEX"] = "1"
t = run_notebook("1_base_models", CODE / "1_base_models/SINCON_final.ipynb", env=env)
S["clean_refined"] = newest([CODE / f"1_base_models/outputs/sincon_{RUN_MODE}_*/submission_clean_refined.csv"], t)
print("1단계 출력:", S["clean_refined"])


--- 코드 셀 1/6 ---
설치 완료. Python: 3.12.15 (main, Oct  3 2026, 00:54:33) [Clang 22.1.3 ]

--- 코드 셀 2/6 ---
모드: assemble 장치: cpu 스레드: 4

--- 코드 셀 3/6 ---
패키지: <실행 폴더>/code/1_base_models
출력 폴더: <실행 폴더>/code/1_base_models/outputs/sincon_assemble_20261113T112904_936662Z

--- 코드 셀 4/6 ---
assemble: 모든 모델 학습을 생략하고 동결 모델 출력을 사용합니다.

--- 코드 셀 5/6 ---
시작 01_early | 로그 01_early.log
완료 01_early: 7.08초
시작 02_events_before_model | 로그 02_events_before_model.log
완료 02_events_before_model: 3.61초
시작 03_model_layers | 로그 03_model_layers.log
완료 03_model_layers: 13.60초
시작 04_events_after_model | 로그 04_events_after_model.log
완료 04_events_after_model: 3.44초

--- 코드 셀 6/6 ---
{
  "rows": 157194,
  "columns": [
    "example_id",
    "boardings"
  ],
  "official_id_order": true,
  "finite_nonnegative": true,
  "max_abs_log1p_difference": 0.0,
  "rms_log1p_difference": 0.0,
  "max_abs_boardings_difference": 0.0,
  "reference_sha256": "519c88bd46e991677316175a2d24f802db6248ebb7d3971ecbe75477532f2199",
  "output_sh

## 2. 2단계 — 과거 검증을 통과한 보정 6개 (`code/2_corrections/SINCON_final_step2_layers.ipynb`)

| 순서 | 보정 | 바뀌는 행 | 값을 정한 방법 |
|---|---|---|---|
| 1 | 타임캡슐 슈퍼콘서트 (7/11, 송도달빛축제공원 21~23시) | 3 | 2025년 같은 공연의 실제 초과 승차의 75% |
| 2 | PLAVE 콘서트 (9/12~13, 문학경기장 21~22시) | 4 | 과거 주경기장 공연 수준 쪽으로 35% |
| 3 | 역별 최근 오차 보정 | 전체 | 기준일 직전 60일의 과거 예측 오차를 역별로 평균해 0.25배 |
| 4 | 금요일 공휴일 (7/17, 10/9) | 2,016 | 과거 금요일 공휴일의 역별 차이, log ±0.05 이내, 0.2배 |
| 5 | PLAVE 22시 | 2 | 과거 공연의 22시 초과 승차 ÷ 관객 수 × 예매 수량 |
| 6 | 부평삼거리역 추석 전날·당일 (9/24~25) | 36 | 과거 명절 당일의 평소 일요일 대비 배율 |

이 폴더와 `2b_monday` 에는 제공 CSV가 들어 있지 않아, 1단계 폴더에 든 같은 파일을 `data/` 에 복사한 뒤 실행한다. 입력이 기준 파일과 1e-9 이내로 같을 때만 진행한다.

In [4]:
for package in ("2_corrections", "2b_monday"):
    (CODE / package / "data").mkdir(exist_ok=True)
    for name in ("train.csv", "test.csv", "metaData.csv"):
        if not (CODE / package / "data" / name).is_file():
            shutil.copyfile(CODE / "1_base_models/payload/team/data" / name, CODE / package / "data" / name)
t = run_notebook("2_corrections", CODE / "2_corrections/SINCON_final_step2_layers.ipynb", overrides={"ROOT": CODE / "2_corrections", "BASELINE_CSV": S["clean_refined"]})
S["night_v4_main"] = newest([d / "sincon_step2_*/submission_night_v4_main.csv" for d in (CODE / "2_corrections/outputs", Path("/content"), Path("/kaggle/working"))], t)
print("2단계 출력:", S["night_v4_main"])


--- 코드 셀 1/11 ---

--- 코드 셀 2/11 ---
패키지: <실행 폴더>/code/2_corrections
공식 CSV: <실행 폴더>/code/2_corrections/data
1단계 입력: submission_clean_refined.csv
{
  "python": "3.12.15",
  "numpy": "2.5.4",
  "pandas": "3.1.1",
  "system": "Darwin",
  "machine": "arm64",
  "processor": "arm",
  "random_seed": "none; deterministic layer arithmetic",
  "gpu_used": false
}

--- 코드 셀 3/11 ---
         layer  enabled  changed_rows  max_abs_log_delta
   timecapsule     True             3           1.462482
plave_existing     True             4           0.500000
    station025     True        157194           0.051717
        friday     True          2016           0.050000
       plave22     True             2           0.478957
     lunar3122     True            36           0.939176

--- 코드 셀 4/11 ---
      layer  enabled  changed_rows  max_abs_log_delta
timecapsule     True             3           1.462482
example_id       date  station_id  hour  timecapsule
   J013371 2026-07-11        3139    21     

## 2b·2c. 3단계가 읽는 두 구성 입력 다시 만들기

- **2b 월요일 대체공휴일** (`code/2b_monday/SINCON_monday.ipynb`): 금요일 공휴일 보정과 같은 식을 8/17, 10/5에 적용한다(0.2배, log ±0.05). 1단계 출력을 입력으로 받는다.
- **2c 코리아컵·걷기대회** (`code/2c_cup_walk/model/build_v4_cup_walk.py`): 코리아컵 4행, 걷기대회 7행. 2단계 출력을 입력으로 받는다.

3단계는 이 두 결과를 폴더 안의 보관본(`code/3_stack/inputs/`)으로 읽는다. 여기서는 다시 만들어 보관본과 같은지 확인한다.

In [5]:
t = run_notebook("2b_monday", CODE / "2b_monday/SINCON_monday.ipynb", overrides={"ROOT": CODE / "2b_monday", "BASELINE_CSV": S["clean_refined"], "OUTPUT_DIR": out / "monday"})
S["monday_main"] = newest([out / "monday/sincon_monday_*/submission_monday_main.csv"], t)

(out / "cup_walk").mkdir(exist_ok=True); S["cup_walk"] = out / "cup_walk/submission_night_v4_main_cup_walk.csv"
run("2c_cup_walk", [sys.executable, "-B", "model/build_v4_cup_walk.py", "--official", CODE / "3_stack/data", "--base", S["night_v4_main"],
                    "--reference", CODE / "3_stack/reference/submission_night_v4_main.csv", "--calendar", "inputs/calendar_features.csv",
                    "--protected", "inputs/protected_event_keys.csv", "--output", S["cup_walk"]], cwd=CODE / "2c_cup_walk")

links = {"월요일 후보 vs 3단계 보관 입력": compare(S["monday_main"], CODE / "3_stack/inputs/submission_monday_main.csv"),
         "코리아컵·걷기 후보 vs 3단계 보관 입력": compare(S["cup_walk"], CODE / "3_stack/inputs/submission_night_v4_main_cup_walk.csv")}
print("보관 입력 대조 (최대 log1p 차이):", links)
assert max(links.values()) <= 1e-9


--- 코드 셀 1/6 ---

--- 코드 셀 2/6 ---
패키지: <실행 폴더>/code/2b_monday

--- 코드 셀 3/6 ---
{
  "accepted": true,
  "acceptance": "sha256_exact",
  "supplied_path": "<실행 폴더>/code/1_base_models/outputs/sincon_assemble_20261113T112904_936662Z/submission_clean_refined.csv",
  "input_sha256": "519c88bd46e991677316175a2d24f802db6248ebb7d3971ecbe75477532f2199",
  "reference_sha256": "519c88bd46e991677316175a2d24f802db6248ebb7d3971ecbe75477532f2199",
  "max_abs_log1p_difference": 0.0,
  "tolerance": 1e-09,
  "numeric_equivalent_input_normalized_to_reference": true,
  "reason": "Preserve the original six layers including their exact-hash input contract; equivalent inputs share the frozen numeric reference."
}

--- 코드 셀 4/6 ---
적용 순서: ('timecapsule', 'plave_existing', 'station025', 'friday', 'monday', 'plave22', 'lunar3122')
행 수: 157194

--- 코드 셀 5/6 ---
월요일을 끈 주후보 재현: {'passed': True, 'max_abs_log1p_difference': 0.0, 'tolerance': 1e-10}

--- 코드 셀 6/6 ---
{
  "success": true,
  "rows": 157194,
  "officia

<실행 폴더>/code/2c_cup_walk/model/build_v4_cup_walk.py:49: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  forecast = pd.read_csv(io.BytesIO(contents), float_precision="round_trip")
<실행 폴더>/code/2c_cup_walk/model/build_v4_cup_walk.py:49: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  forecast = pd.read_csv(io.BytesIO(contents), float_precision="round_trip")
157194 rows, 11 changes, SHA-256 93dee18c6e3484c7837a53082920eada9ae9d2506045e9b7e8c262c5bb07a569

[2c_cup_walk] 0.7초
보관 입력 대조 (최대 log1p 차이): {'월요일 후보 vs 3단계 보관 입력': 0.0, '코리아컵·걷기 후보 vs 3단계 보관 입력': 8.881784197001252e-16}


## 3. 3단계 — 보정 7개 단계 (`code/3_stack/SINCON_final_stack.ipynb`)

| 단계 | 내용 | 계수 |
|---|---|---|
| 1 K1 | 일반일의 역×요일×시간대 오차 보정 | 최근 84일, 수축 n/(n+12), 0.25배, 최소 6일 |
| 2 K3 | K1 + 월요일 대체공휴일 2,016행 + 코리아컵 4행·걷기대회 7행 | 위 2b·2c |
| 3 | 검단호수공원역 수준, PLAVE 공연 중 4행 | 과거 오차로 적합 |
| 4 | 계절 재적합 | K1 적용 후 남은 과거 오차로 다시 계산 |
| 5 L1 | K1 규칙을 제외했던 7개 역으로 확장 | K1과 같음 |
| 6 | 시간대 고주파 패턴 | 실측 프로필의 고주파 성분 15% |
| 7 | 7개 역 일반일에 `0.1 × (XGBoost − 대체 앙상블)` | 보관한 모델 예측 6개(`predictions/`) |

단계마다 출력의 ID·순서와 기준 파일과의 최대 log1p 차이를 출력한다. 실측이 든 입력의 최대 날짜가 2026-06-30 이하인지도 검사한다.

In [6]:
run_notebook("3_stack", CODE / "3_stack/SINCON_final_stack.ipynb",
             env={"SINCON_STACK_ROOT": CODE / "3_stack", "SINCON_STACK_OUTPUT": out / "stack", "SINCON_STACK_BASELINE": S["night_v4_main"]})
S["xgb7"] = out / "stack/submission_xgb7_w10_on_curvature.csv"
print("3단계 출력:", S["xgb7"])


--- 코드 셀 1/11 ---

--- 코드 셀 2/11 ---
Python 3.12.15 | pandas 3.1.1 | NumPy 2.5.4
패키지 <실행 폴더>/code/3_stack | 출력 <실행 폴더>/run_outputs/stack

--- 코드 셀 3/11 ---
data/train.csv | 최대 날짜 date=2026-06-30 | 승하차 실측 또는 실측 log 포함: 2026-06-30 이하 확인
data/test.csv | 최대 날짜 date=2026-10-31 | 예측 대상 키, 승하차 실측 없음
cache/hourbias_history_oof.csv.gz | 최대 날짜 date=2026-06-30; origin=2026-04-30 | 승하차 실측 또는 실측 log 포함: 2026-06-30 이하 확인
cache/calendar_flags.csv | 최대 날짜 date=2026-10-31 | 달력·행사 범위·역 메타데이터, 승하차 실측 없음
cache/events_frozen_v3.csv | 최대 날짜 date=2026-10-17 | 달력·행사 범위·역 메타데이터, 승하차 실측 없음
cache/protected_event_keys.csv | 최대 날짜 date=2026-10-31 | 달력·행사 범위·역 메타데이터, 승하차 실측 없음
cache/downstream_oof_nested_main.csv.gz | 최대 날짜 date=2026-06-30; origin=2026-04-30 | 승하차 실측 또는 실측 log 포함: 2026-06-30 이하 확인
cache/calendar_features.csv | 최대 날짜 date=2026-10-31 | 달력·행사 범위·역 메타데이터, 승하차 실측 없음
cache/events_daily.csv | 최대 날짜 date=2026-10-25 | 달력·행사 범위·역 메타데이터, 승하차 실측 없음
cache/festival_station_days.csv | 최대 날짜 date=2026-10-27 | 달력·

## 4. 4단계 — 다른 노선 자료를 쓰는 세 층 (`code/4_rail/SINCON_rail_factors.ipynb`)

입력은 7호선과 환승역을 뺀 다른 노선 544개 역의 일별 승차다(서울 열린데이터광장 OA-12914). 2026년 7~10월 값을 쓰며, 규칙과의 관계는 `docs/DISCLOSURE.md` 2절에 적었다. 노트북이 자료에 7호선과 환승역 행이 없는지 다시 검사한다.

| 층 | 방법 | 계수 |
|---|---|---|
| 변화 옮기기 | 타 노선의 "같은 요일 직전 8주 중앙값 대비 증감"에서 주성분을 구하고, 대상 역마다 자기 증감을 주성분에 Ridge로 적합한다. 시험 기간 날짜마다 "예상 변화 − 우리 예측의 변화"의 일부를 더한다 | 주성분 30개, α=5, 일반일 0.2·공휴일류 0.3, 행당 ±0.15 |
| 공휴일류 수준 | 인접 노선 날짜 피처를 넣은 XGBoost와 넣지 않은 XGBoost의 차이(역·날짜 평균)를 공휴일류 날짜에 더한다 | 0.5배, ±0.15 |
| 시간대별 반응 | 역·시간대마다 증감을 주성분에 적합해 "예상 형태 변화 − 우리 예측의 형태 변화"를 더한다 | 주성분 12개, α=5, 0.1배, ±0.1 |

계수는 모두 2026-06-30 이전 자료의 과거 9개 기준일 검증으로 정했다(`docs/METHOD.md` 6절, 검증 스크립트 `code/4_rail/validation/`).

In [7]:
run_notebook("4_rail", CODE / "4_rail/SINCON_rail_factors.ipynb", overrides={"ROOT": CODE / "4_rail", "BASELINE_CSV": S["xgb7"], "OUTPUT_DIR": out / "rail"})
S["k30_hour"] = out / "rail/submission_rail_factors_k30_hour_on_xgb7.csv"
print("4단계 출력:", S["k30_hour"])


--- 코드 셀 1/7 ---

--- 코드 셀 2/7 ---
python 3.12.15 | pandas 3.1.1 | numpy 2.5.4
패키지: <실행 폴더>/code/4_rail
출력: <실행 폴더>/run_outputs/rail

--- 코드 셀 3/7 ---
타 노선 자료: 371179 행, 26 개 노선, 561 개 역, 2025-01-01 ~ 2026-10-31 (7~10월 실측 포함: 구두 허용 전제)
제외한 이름 수: 52 | 공식 학습 자료 마지막 날짜: 2026-06-30
기반 입력: <실행 폴더>/run_outputs/stack/submission_xgb7_w10_on_curvature.csv | 기준 파일과 최대 log 차이 8.881784197001252e-16
바꾸지 않는 역·날짜: 357 | 공휴일류 날짜(시험 기간): 10

--- 코드 셀 4/7 ---
쓴 다른 노선 역: 544 | 학습 날짜 수: 539 | 주성분 설명 비율: 0.96
바뀐 행: 154458 | 바꾸지 않은 행: 2736 | 최대 log 변화: 0.15
    다른 노선으로 본 예상 변화  우리 예측의 변화  더한 양(평균)
월                                       
7           -0.0713    -0.0998   0.00641
8           -0.1049    -0.1307   0.00635
9           -0.0521    -0.0861   0.00740
10          -0.0203    -0.0562   0.00788

--- 코드 셀 5/7 ---
공휴일 수준 층이 바꾼 행: 12546 | 최대 log 변화: 0.15
       날짜별 평균 log 변화
07-17         0.0135
08-15        -0.0308
08-17         0.0270
09-24        -0.0944
09-25        -0.0585
09-26        -0.0303
09-27 

## 5. 최종 예측 저장과 제출 파일 비교

4단계 출력을 `submission_final_main.csv` 로 복사하고 `predictions/submission_final_main.csv`(Kaggle에 올린 파일과 바이트 동일)와 비교한다. log1p 예측의 최대 차이가 1e-9 이하면 재현된 것으로 본다.

In [8]:
final = out / "submission_final_main.csv"; shutil.copyfile(S["k30_hour"], final)
submitted = project / "predictions/submission_final_main.csv"
gap = compare(final, submitted); same_bytes = sha256(final) == sha256(submitted)
result = pd.read_csv(final)
print("최종 예측:", final)
print("행 수", len(result), "| 열", list(result.columns), "| 결측", int(result.isna().sum().sum()), "| 최솟값", float(result.iloc[:, -1].min()))
print("SHA-256 (이번 실행)", sha256(final)); print("SHA-256 (제출 파일) ", sha256(submitted))
print("제출 파일과 최대 log1p 차이:", gap, "| 바이트 일치:", same_bytes)
summary = {"mode": RUN_MODE, "device": DEVICE, "python": platform.python_version(), "platform": platform.platform(), "numpy": np.__version__, "pandas": pd.__version__,
           "seconds": T, "total_seconds": round(sum(T.values()), 1), "outputs": {k: str(v) for k, v in S.items()}, "archived_input_checks_max_abs_log1p_diff": links,
           "final_prediction": {"output": str(final), "sha256": sha256(final), "submitted_sha256": sha256(submitted), "byte_identical": same_bytes, "max_abs_log1p_diff_vs_submitted": gap}}
(out / "run_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(pd.Series(T, name="걸린 시간(초)").to_frame().T.to_string(index=False), "| 합계", summary["total_seconds"], "초")
assert gap <= 1e-9, "최종 예측이 제출 파일과 수치로 다르다."

최종 예측: <실행 폴더>/run_outputs/submission_final_main.csv
행 수 157194 | 열 ['example_id', 'boardings'] | 결측 0 | 최솟값 2.602686342274018
SHA-256 (이번 실행) cf786c87a06c1efa5a6fbbd96e782ea65657409485a2a50f65936ecc14ccc51e
SHA-256 (제출 파일)  aa455744556514d54ff2f3fe142117f331a04aa37141e59e6daf1cbfe0e8469c
제출 파일과 최대 log1p 차이: 8.881784197001252e-16 | 바이트 일치: False
 1_base_models  2_corrections  2b_monday  2c_cup_walk  3_stack  4_rail
          29.0            2.5        3.4          0.7      4.3     1.6 | 합계 41.5 초
